# ⚡ ShadowVault Interactive Quickstart

[![GitHub stars](https://img.shields.io/github/stars/dhirajparihar/shadow-vault?style=social)](https://github.com/dhirajparihar/shadow-vault)
[![PyPI](https://img.shields.io/pypi/v/shadow-vault.svg)](https://pypi.org/project/shadow-vault/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

**ShadowVault** is a bi-directional, format-preserving reversible PII proxy for AI agents, LangChain/LangGraph, and MCP tools.

### Why ShadowVault?
* 🔒 **Zero Leaks to Cloud LLMs**: PII is replaced with synthetically valid, format-preserving tokens *before* prompts leave your boundary.
* ⚙️ **Zero Code Changes to Tools**: Local tools decorated with `@vault_guard` receive real PII automatically.
* 🎩 **Format-Preserving Tokens**: Synthetic cards pass Luhn Mod 10, Aadhaar numbers pass Verhoeff D5, emails match RFC 5322. Downstream parsers and validators never break.
* ⚡ **Streaming Holdback Buffer**: Eliminates torn-token leaks across OpenAI/Anthropic Server-Sent Events (SSE) micro-chunks.
* 🧰 **Zero-Dependency Core**: Runs on pure Python standard library (`sqlite3`, `hmac`, `hashlib`, `re`). Optional `cryptography` enables hardware-accelerated AES-256-GCM.

## 1. Installation
Install the core package from PyPI:

In [ ]:
!pip install --upgrade shadow-vault cryptography

## 2. Initialize ShadowVault with AES-256-GCM Encryption
All token-to-PII mappings are encrypted at rest with authenticated AES-256-GCM and indexed with deterministic blind HMAC indexes.

In [ ]:
from shadow_vault import ShadowVault

# Initialize vault with authenticated encryption
vault = ShadowVault(
    encryption_key="super-secret-vault-master-key-32b!",
    require_encryption=True,
    fail_closed=True,
)
print("Vault initialized successfully with authenticated encryption!")

## 3. Inbound Prompt Dehydration (Format-Preserving Tokens)
When a user prompt contains sensitive data (emails, cards, Aadhaar, PAN, UPI, SSN, API keys), ShadowVault replaces them with synthetically valid tokens.

In [ ]:
user_prompt = (
    "Customer Priya Sharma (email: priya.sharma@fintech.co, phone: +91 98765 43210) "
    "reported unauthorized charge of $150 on Visa 4532 0151 1283 0366. "
    "Her verified Aadhaar is 9999 0000 1234 and UPI is priya@okhdfcbank."
)

session_id = "support-session-user-8812"
dehydrated_prompt = vault.dehydrate(user_prompt, session_id=session_id)

print("--- ORIGINAL USER PROMPT (Kept Local) ---")
print(user_prompt)
print("\n--- DEHYDRATED PROMPT (Sent to Cloud LLM) ---")
print(dehydrated_prompt)

### Format Validation Check
Notice how the synthetic credit card passes Luhn Mod-10, and the synthetic Aadhaar passes Verhoeff checksums:

In [ ]:
from shadow_vault import is_luhn_valid, is_verhoeff_valid
import re

# Extract the synthetic credit card
card_match = re.search(r'\b4[0-9]{15}\b', dehydrated_prompt)
if card_match:
    card = card_match.group(0)
    print(f"Synthetic Card: {card} -> Luhn Valid: {is_luhn_valid(card)}")

# Extract the synthetic Aadhaar
aadhaar_match = re.search(r'\b[2-9][0-9]{3}\s[0-9]{4}\s[0-9]{4}\b', dehydrated_prompt)
if aadhaar_match:
    aadhaar = aadhaar_match.group(0)
    print(f"Synthetic Aadhaar: {aadhaar} -> Verhoeff Valid: {is_verhoeff_valid(aadhaar)}")

## 4. Local Tool Protection with `@vault_guard`
When an AI agent executes a tool (e.g. charging a card, querying a database, or sending an email), `@vault_guard` intercepts the synthetic arguments and restores the genuine PII locally. The LLM never sees the real values.

In [ ]:
from shadow_vault import vault_guard

@vault_guard(vault=vault, session_id_arg="session_id")
def charge_payment_gateway(customer_email: str, card_number: str, amount_usd: float, session_id: str):
    """Local tool that executes against real payment APIs."""
    print(f"[GATEWAY EXECUTION - LOCAL PRIVACY BOUNDARY]")
    print(f"  Target Customer:  {customer_email}")
    print(f"  Real Card Number: {card_number}")
    print(f"  Amount:           ${amount_usd:.2f}")
    return f"Successfully charged ${amount_usd:.2f} to {customer_email} on card {card_number[-4:]}"

# Simulate the agent calling the tool with synthetic tokens received from LLM
records = vault.store.get_session_records(session_id)
synthetic_email = [r.synthetic_token for r in records if r.entity_type == "email"][0]
synthetic_card = [r.synthetic_token for r in records if r.entity_type == "credit_card"][0]

print(f"LLM calling tool with synthetic arguments: email={synthetic_email}, card={synthetic_card}\n")
result = charge_payment_gateway(
    customer_email=synthetic_email,
    card_number=synthetic_card,
    amount_usd=150.00,
    session_id=session_id,
)
print(f"Tool Output (safe for LLM): {result}")

## 5. Streaming Holdback Buffer (Solving Torn SSE Chunks)
When LLMs stream tokens via Server-Sent Events (SSE), a synthetic token like `u_92a7e1@shadow.vault` can be split across arbitrary chunk boundaries (e.g. `['u_', '92a', '7e1@', 'shadow.vault']`).

`StreamHydrator` uses an atomic lookahead sliding buffer to hold back partial token candidates and emit restored values smoothly without leaking synthetic artifacts.

In [ ]:
from shadow_vault import StreamHydrator

hydrator = StreamHydrator(vault=vault, session_id=session_id)

# Simulate fragmented chunks from an OpenAI streaming endpoint
micro_chunks = [
    "Dear ",
    "customer, ",
    "a confirmation ",
    "receipt was sent to ",
    synthetic_email[:4],         # 'u_92'
    synthetic_email[4:10],       # 'a7e1@'
    synthetic_email[10:],        # 'shadow.vault'
    ". Have a great day!",
]

print("Streaming output to client:")
output_stream = []
for chunk in micro_chunks:
    hydrated_part = hydrator.transform_chunk(chunk)
    if hydrated_part:
        print(hydrated_part, end="", flush=True)
        output_stream.append(hydrated_part)

final_part = hydrator.flush()
if final_part:
    print(final_part, end="", flush=True)
    output_stream.append(final_part)
print("\n\nComplete Stream Successfully Hydrated!")

## 6. Run the Built-In Reproducible Security Audit
Verify all 6 core cryptographic and privacy guarantees in real-time:

In [ ]:
from shadow_vault.verify import run_verification

run_verification()

## Next Steps
- Explore the [Claude Desktop MCP Cookbook](https://github.com/dhirajparihar/shadow-vault/blob/main/examples/cookbooks/claude_desktop_mcp.md)
- Check the [LangGraph Multi-Agent Customer Support Example](https://github.com/dhirajparihar/shadow-vault/blob/main/examples/cookbooks/langgraph_customer_support.py)
- Read our detailed [Threat Model & Security Policy](https://github.com/dhirajparihar/shadow-vault/blob/main/THREAT_MODEL.md)
- Star the repository on [GitHub](https://github.com/dhirajparihar/shadow-vault)!